# Mobile Export + Latency Benchmark
Export the trained mosquito classifier to ONNX, convert to TFLite (Android) with
INT8 quantization, and benchmark size + latency — the metrics that actually matter
for VectorCam's phone-based deployment target.

**Run in Colab.** CPU runtime is fine for this notebook (no training happens here).


## 1. Install export/conversion tools

In [ ]:
!pip install -q onnx onnx2tf onnxruntime tensorflow


## 2. Upload your trained checkpoint

In [ ]:
from google.colab import files
uploaded = files.upload()  # select mosquito_multihead_effnetb0_final.pt
CKPT_PATH = list(uploaded.keys())[0]
print("Using checkpoint:", CKPT_PATH)


## 3. Rebuild the model architecture and load weights

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

class MosquitoMultiHead(nn.Module):
    def __init__(self, n_species, n_sex):
        super().__init__()
        backbone = efficientnet_b0(weights=None)
        self.trunk = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        feat_dim = backbone.classifier[1].in_features

        self.species_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(feat_dim, n_species))
        self.sex_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(feat_dim, n_sex))

    def forward(self, x):
        feats = self.pool(self.trunk(x)).flatten(1)
        return self.species_head(feats), self.sex_head(feats)   # tuple output for ONNX export


ckpt = torch.load(CKPT_PATH, map_location="cpu")
species_map = ckpt["species_map"]
sex_map = ckpt["sex_map"]
IMG_SIZE = ckpt.get("img_size", 224)

model = MosquitoMultiHead(n_species=len(species_map), n_sex=len(sex_map))
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded model: {len(species_map)} species, {len(sex_map)} sex classes, img_size={IMG_SIZE}")


## 4. Export to ONNX
This is the universal intermediate format — both TFLite and CoreML conversion
paths go through it.


In [ ]:
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

torch.onnx.export(
    model,
    dummy_input,
    "mosquito_model.onnx",
    input_names=["image"],
    output_names=["species_logits", "sex_logits"],
    dynamic_axes={"image": {0: "batch"}},
    opset_version=17,
)

import os
print(f"ONNX model size: {os.path.getsize('mosquito_model.onnx') / 1e6:.2f} MB")


## 5. Sanity check the ONNX export
Before converting further, confirm ONNX gives the same output as the original
PyTorch model on the same input — if this doesn't match, don't bother converting
downstream formats until it's fixed.


In [ ]:
import onnxruntime as ort
import numpy as np

torch_species, torch_sex = model(dummy_input)

ort_session = ort.InferenceSession("mosquito_model.onnx")
onnx_out = ort_session.run(None, {"image": dummy_input.numpy()})

species_diff = np.abs(torch_species.detach().numpy() - onnx_out[0]).max()
sex_diff = np.abs(torch_sex.detach().numpy() - onnx_out[1]).max()

print(f"Max diff (species logits): {species_diff:.6f}")
print(f"Max diff (sex logits):     {sex_diff:.6f}")
print("Should both be very close to 0 (< 1e-4). If not, stop and debug before continuing.")


## 6. Convert ONNX -> TFLite (Android target)

In [ ]:
!onnx2tf -i mosquito_model.onnx -o tflite_output -oiqt
# -oiqt also outputs an INT8-quantized version automatically where possible

import os
print("Files produced:")
for f in os.listdir("tflite_output"):
    size_mb = os.path.getsize(os.path.join("tflite_output", f)) / 1e6
    print(f"  {f}  ({size_mb:.2f} MB)")


## 7. Manual INT8 quantization (if the automatic one above didn't produce it)
INT8 quantization is the step that most affects real phone latency and app size —
worth doing explicitly with a representative dataset for best accuracy retention.


In [ ]:
import tensorflow as tf
import numpy as np

# Representative dataset generator — ideally use a handful of REAL validation
# images here instead of random noise, for better quantization accuracy.
# If you have your labels.csv + image folder available in this Colab session,
# swap this out for real preprocessed images.
def representative_dataset():
    for _ in range(100):
        data = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
        yield [data]

converter = tf.lite.TFLiteConverter.from_saved_model("tflite_output")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

tflite_int8_model = converter.convert()

with open("mosquito_model_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

print(f"INT8 TFLite size: {os.path.getsize('mosquito_model_int8.tflite') / 1e6:.2f} MB")


## 8. Benchmark latency
This is the number that actually answers "is this fast enough for VectorCam."
Compares the original float32 ONNX model against the INT8 TFLite model.


In [ ]:
import time
import numpy as np

def benchmark_onnx(session, n_runs=50):
    dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    # warmup
    for _ in range(5):
        session.run(None, {"image": dummy})
    times = []
    for _ in range(n_runs):
        start = time.time()
        session.run(None, {"image": dummy})
        times.append((time.time() - start) * 1000)  # ms
    return np.mean(times), np.std(times)


def benchmark_tflite(interpreter, n_runs=50):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_shape = input_details[0]['shape']
    dummy = np.random.rand(*input_shape).astype(input_details[0]['dtype'])

    for _ in range(5):
        interpreter.set_tensor(input_details[0]['index'], dummy)
        interpreter.invoke()

    times = []
    for _ in range(n_runs):
        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], dummy)
        interpreter.invoke()
        times.append((time.time() - start) * 1000)
    return np.mean(times), np.std(times)


onnx_mean, onnx_std = benchmark_onnx(ort_session)

interpreter = tf.lite.Interpreter(model_path="mosquito_model_int8.tflite")
interpreter.allocate_tensors()
tflite_mean, tflite_std = benchmark_tflite(interpreter)

print("=" * 50)
print("LATENCY BENCHMARK (Colab CPU — phone will differ, but ratio holds roughly)")
print("=" * 50)
print(f"{'Format':<20} {'Mean (ms)':<12} {'Std (ms)':<10} {'Size (MB)'}")
print(f"{'ONNX (float32)':<20} {onnx_mean:<12.2f} {onnx_std:<10.2f} {os.path.getsize('mosquito_model.onnx')/1e6:.2f}")
print(f"{'TFLite (INT8)':<20} {tflite_mean:<12.2f} {tflite_std:<10.2f} {os.path.getsize('mosquito_model_int8.tflite')/1e6:.2f}")
print(f"\nSpeedup: {onnx_mean/tflite_mean:.2f}x")
print(f"Size reduction: {(os.path.getsize('mosquito_model.onnx')/os.path.getsize('mosquito_model_int8.tflite')):.2f}x")


## 9. Decision point

- **If INT8 TFLite latency is roughly 50-150ms or less**, that's a reasonable
  real-time bar for a handheld scanning tool — no further architecture change needed.
- **If it's much slower**, the next step is swapping the backbone to MobileNetV3-Small
  (same two-head wrapper, same training loop, ~2-3x faster on mobile NPUs at a
  modest accuracy cost) rather than assuming EfficientNet-B0 is the ceiling.

Either way, download both exported files for your repo/writeup:


In [ ]:
from google.colab import files
files.download("mosquito_model.onnx")
files.download("mosquito_model_int8.tflite")


## Notes for your writeup

- Colab CPU timing is not the same as an actual Android/iOS device, but the
  **relative** speedup between float32 and INT8 is informative even off-device.
- For a real device latency number, the TFLite model needs to be run inside an
  actual Android app (via the TFLite Android library) or benchmarked with
  `adb shell` + the `benchmark_model` tool from the TensorFlow Lite repo.
- CoreML (iOS) follows the same ONNX starting point:
  `import coremltools as ct; ct.convert("mosquito_model.onnx", convert_to="mlprogram")`
